# 01. Dataset audit

This notebook reproduces the audit of `docs/02_data_audit.md`. It reads the raw files downloaded by `python -m hatari.data.download` and the corpus built by `python -m hatari.data.corpus`.

**Precautions**

- The raw files are checked against their SHA-256 checksums before any statistic is computed.
- The test part of the corpus is only counted here, its texts are not read.

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def run_tests(*files):
    result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", *files], cwd=ROOT, capture_output=True, text=True)
    print(result.stdout.strip().splitlines()[-1])

## 1. Unit tests

The tests of the loaders, of the audit keys, of the cleaning of generator markers and of the corpus split.

In [2]:
run_tests("tests/test_sources.py", "tests/test_audit.py", "tests/test_cleaning.py", "tests/test_corpus.py")

16 passed in 2.55s


## 2. Integrity of the raw files

In [3]:
from hatari.data.download import RAW_DIR, SOURCES, sha256_of

display(pd.DataFrame([{"source": s.name, "file": s.filename, "expected": s.sha256,
                       "matches": sha256_of(RAW_DIR / s.filename) == s.sha256} for s in SOURCES]))

,source,file,expected,matches
0,uci_sms_spam,uci_sms_spam.zip,1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3,True
1,mendeley_smishing,mendeley_smishing.zip,9bbf3188fdad81495d8e82825648b9b63b53fc86841a3d26c02629990b233cc3,True
2,french_translated,french_translated.csv,ebfccd04514e184440a8de53b8ca57fc9bc300c7ae0f44417923d7e019ef093a,True
3,french_synthetic,french_synthetic.csv,8d8f9130838a0bea10bc4f99dbe563b2ca903f90be38aa13a9a57c46e33aac4d,True


## 3. Class distribution and quality of each source

In [4]:
from hatari.data.audit import generator_markers, overlap, split_leakage, summary, translated_french_defects
from hatari.data.sources import load_all

sources = load_all()
display(pd.DataFrame({name: summary(frame) for name, frame in sources.items()}).T)

,messages,suspicious,suspicious_share,labels,channels,duplicate_messages,distinct_texts,distinct_templates,binary_label_conflicts,replacement_characters,median_characters
uci_sms_spam,5574,747,0.134,"{'ham': 4827, 'spam': 747}",{'sms': 5574},447,5127,5094,0,0,61
mendeley_smishing,5971,1127,0.1887,"{'ham': 4844, 'smishing': 638, 'spam': 489}",{'sms': 5971},147,5824,5701,0,145,66
french_synthetic,10000,5000,0.5,"{'spam': 5000, 'legitimate': 5000}","{'sms': 5000, 'email': 5000}",0,10000,441,0,0,145


**Findings**

- UCI has 5,574 messages, 13.4% suspicious, with 447 duplicates. Mendeley has 5,971 messages, 18.9% suspicious, with 145 texts damaged by replacement characters.
- The French synthetic set has 10,000 distinct messages but only 441 distinct texts once digits are masked.
- No text carries both a legitimate and a suspicious label within a source.

## 4. Overlap of the two English sources

In [5]:
shared = overlap(sources["uci_sms_spam"], sources["mendeley_smishing"])
print({key: value for key, value in shared.items() if key != "label_crosstab"})
display(pd.DataFrame(shared["label_crosstab"]).fillna(0).astype(int).rename_axis(index="UCI", columns="Mendeley"))

{'shared_texts': 4943, 'only_first': 184, 'only_second': 881}


Mendeley,ham,smishing,spam
UCI,,,
ham,4367,0,0
spam,0,267,309


**Findings**

- 4,943 distinct texts are in both sources, so they must be merged and deduplicated before any split.
- The binary labels always agree. Mendeley splits the UCI spam into spam and smishing, inconsistently, so only the binary target is learned.

## 5. Label leakage in the French synthetic set

In [6]:
synthetic = sources["french_synthetic"]
display(pd.Series(generator_markers(synthetic)).to_frame("value"))
display(pd.Series(split_leakage(synthetic)).to_frame("value"))

,value
legitimate_marker_by_label,"{False: 1.0, True: 0.0}"
spam_marker_by_label,"{False: 0.0, True: 1.0}"
distinct_cleaned_texts,161
distinct_cleaned_texts_by_label,"{False: 77, True: 84}"
cleaned_with_digits_by_label,"{False: 0.3, True: 0.0}"


,value
texts_in_several_splits,0
templates_in_several_splits,0


**Findings**

- Every legitimate message carries a `[référence SM-…]` or `EM-…` marker and every spam message a `Dossier Ville-1234` fragment. Both are removed before use.
- Once they are removed, 161 distinct texts remain, 77 legitimate and 84 suspicious. After cleaning, 30% of legitimate texts contain digits and no suspicious text does, the opposite of real fraud.
- The published split separates the template families, with no text or digit template shared between splits.

## 6. Translated French set, excluded

In [7]:
display(pd.Series(translated_french_defects()).to_frame("value"))

,value
non_empty_lines,5585
lines_with_a_label,5535
labels,"{'ham': 4792, 'spam': 743}"
lines_without_a_label,50
examples_without_a_label,"[Voici la traduction complète des messages demandés (seulement la deuxième colonne) :, ```, (Et mon homme Carlos pas..."


**Finding.** 50 lines have no label, among them text left by the translation tool. With 5,535 labelled lines against 5,574 in UCI, the file cannot be aligned with its source, and it would leak paraphrases of UCI messages into the test set.

## 7. Corpus and split

In [8]:
corpus = pd.read_csv(ROOT / "data" / "processed" / "corpus.csv")
display(corpus.groupby(["language", "origin", "split"])["suspicious"].agg(messages="size", suspicious="sum", share="mean").round(3))
for language, part in corpus.groupby("language"):
    print(language, "| groups in both splits:", int(part.groupby("group")["split"].nunique().gt(1).sum()))

messages  suspicious  share
language origin    split                             
en       real      test       1196         209  0.175
                   train      4812         836  0.174
fr       synthetic test         44          14  0.318
                   train       117          70  0.598

en | groups in both splits: 0
fr | groups in both splits: 0


**Findings**

- The English corpus has 6,008 distinct texts, 4,812 in training and 1,196 in test, with 17.4% suspicious in each.
- The French part has 117 training and 44 test texts.
- No group is split between training and test.